### **Estrategias para la Selección del Siguiente Token**

La función `torch.multinomial` no es la única herramienta para seleccionar tokens. Una vez que obtenemos la distribución de probabilidades sobre el vocabulario mediante Softmax, se abre un abanico de estrategias de decodificación (*decoding strategies*) para elegir el siguiente token.

En esta lección nos centramos en **3 métodos fundamentales**:

1. **Greedy Search (Búsqueda Golosa / Argmax):** Selecciona siempre y sin excepción el token con la mayor probabilidad. Es un método determinista y directo.
2. **Multinomial Sampling (Muestreo Ponderado):** Muestrea al azar siguiendo la distribución de probabilidades generada por Softmax (pudiendo ajustar su diversidad con la Temperatura).
3. **Top-K Sampling:** Restringe las opciones a los $K$ tokens con mayor probabilidad y vuelve a calcular las probabilidades dentro de ese subconjunto antes de muestrear.

---

### **Otros métodos esenciales en LLMs:**

Además de estas tres técnicas, existen otros enfoques ampliamente utilizados en la industria:

* **Top-p / Nucleus Sampling:** En lugar de fijar un número rígido de tokens ($K$), fija un umbral de probabilidad acumulada $p$ (por ejemplo, $p = 0.90$). Selecciona dinámicamente el grupo mínimo de tokens cuya suma alcanza el $90\%$ de probabilidad antes de muestrear.
* **Beam Search (Búsqueda por Haz):** Evalúa múltiples rutas simultáneamente. Mantiene en memoria las $N$ mejores secuencias probables paso a paso, ideal para tareas de traducción automática o resúmenes donde la coherencia global importa más que la creatividad inmediata.
* **Muestreo con Penalizaciones (Repetition & Frequency Penalties):** Aplica descuentos sobre los *logits* de palabras que ya se han generado para evitar que el modelo entre en bucles o repita expresiones.
* **Contrastive Search:** Combina la probabilidad otorgada por el modelo con un cálculo de similitud con los tokens previos, evitando que el modelo genere texto redundante o genérico.

---


In [ ]:
# Veremos Top-p - Top-k-sampling - Greedy

**Greedy Search (Búsqueda Golosa / Argmax):** Selecciona siempre, sin excepción, el token que tenga la mayor probabilidad asignada.

---

### **Frase de entrada:**

> *"El desarrollador tomó un sorbo de su caliente taza de _____"*

---

### **Distribución de probabilidades de los posibles tokens:**

| Token (Palabra) | Probabilidad | Selección |
| --- | --- | --- |
| **café** | **0.58** | 🟢 **Elegido (Greedy)** |
| té | 0.26 | ❌ |
| agua | 0.11 | ❌ |
| teclado | 0.04 | ❌ |
| universo | 0.01 | ❌ |

---

**Resultado final generado:**

> *"El desarrollador tomó un sorbo de su caliente taza de **café**."*

**¿Por qué lo elige?**

No importa que *"té"* ($26\%$) o *"agua"* ($11\%$) sean opciones completamente coherentes; la estrategia **Greedy** ignora cualquier alternativa y se queda de forma determinista con el pico más alto ($58\%$).

In [ ]:
# -----------------------------

Podemos contrastarlo con TOP-K

**Top-K Sampling:** Filtra el vocabulario para conservar únicamente los **$K$ tokens con mayor probabilidad**, descarta el resto y vuelve a calcular las probabilidades de ese grupo reducido antes de realizar el muestreo aleatorio.

---

### **Frase de entrada:**

> *"El desarrollador tomó un sorbo de su caliente taza de _____"*

---

### **Paso 1: Filtrado de las mejores $K$ opciones (para $K = 3$)**

De la lista original de 5 palabras, tomamos únicamente las **3 superiores** y eliminamos las opciones absurdas o con muy baja probabilidad (*ruido*):

| Token (Palabra) | Probabilidad Original | Estado en $K = 3$ |
| --- | --- | --- |
| **café** | **0.58** | 🟢 Conservado |
| **té** | **0.26** | 🟢 Conservado |
| **agua** | **0.11** | 🟢 Conservado |
| ~~teclado~~ | ~~0.04~~ | 🔴 *Descartado* |
| ~~universo~~ | ~~0.01~~ | 🔴 *Descartado* |

---

### **Paso 2: Re-normalización de las probabilidades**

Como eliminamos dos palabras, las probabilidades de los 3 tokens restantes ya no suman 1.0 ($0.58 + 0.26 + 0.11 = 0.95$). Dividimos cada una entre $0.95$ para renormalizarlas a $100\%$:

* **café:** $0.58 / 0.95 \approx$ **$61.1\%$**
* **té:** $0.26 / 0.95 \approx$ **$27.4\%$**
* **agua:** $0.11 / 0.95 \approx$ **$11.5\%$**

---

### **Paso 3: Selección por muestreo**

A diferencia de Greedy Search (que siempre elegirá *"café"*), **Top-K realiza un muestreo aleatorio ponderado** sobre estas 3 opciones renormalizadas:

* **$61.1\%$** de las veces elegirá **"café"** (alta coherencia).
* **$27.4\%$** de las veces elegirá **"té"** (buena variedad semántica).
* **$11.5\%$** de las veces elegirá **"agua"** (opción secundaria válida).
* **$0\%$** de las veces elegirá **"teclado"** o **"universo"** (se eliminó por completo el riesgo de alucinaciones raras).

---

### **Diferencia clave entre ambas estrategias:**

| Propiedad | Greedy Search | Top-K Sampling ($K=3$) |
| --- | --- | --- |
| **Comportamiento** | Determinista (100% predecible). | Aleatorio controlado. |
| **Diversidad** | Cero (siempre genera la misma respuesta). | Alta (puede dar distintas variantes válidas). |
| **Riesgo de texto plano/repetitivo** | Alto. | Muy bajo. |

In [ ]:
# Si el parametro K=2 solo seleccionara un 50% 50% de estos tokens (el parametro K es el numero de tokens)

**Top-p Sampling (Muestreo de Núcleo / *Nucleus Sampling*):** En lugar de fijar un número rígido de candidatos (como el $K$ en Top-K), fija un umbral de **probabilidad acumulada $p$** (por ejemplo, $p = 0.90$ o $90\%$).

El modelo selecciona el grupo dinámico más pequeño de tokens cuya suma alcanza o supera ese $90\%$, descarta el resto y realiza el muestreo dentro de ese "núcleo".

---

### **Frase de entrada:**

> *"El desarrollador tomó un sorbo de su caliente taza de _____"*

---

### **Paso 1: Acumular probabilidades en orden descendente (para $p = 0.90$)**

Ordenamos las palabras de mayor a menor probabilidad y sumamos acumulativamente hasta alcanzar el $90\%$ ($0.90$):

| Token (Palabra) | Probabilidad Individual | Probabilidad Acumulada | Estado ($p = 0.90$) |
| --- | --- | --- | --- |
| **café** | **0.58** | $0.58$ | 🟢 **Incluido** ($58\% < 90\%$) |
| **té** | **0.26** | $0.58 + 0.26 = \mathbf{0.84}$ | 🟢 **Incluido** ($84\% < 90\%$) |
| **agua** | **0.11** | $0.84 + 0.11 = \mathbf{0.95}$ | 🟢 **Incluido** (Alcanza el $95\% \ge 90\%$) |
| ~~teclado~~ | ~~0.04~~ | $0.95 + 0.04 = 0.99$ | 🔴 *Descartado (Corte aplicado)* |
| ~~universo~~ | ~~0.01~~ | $0.99 + 0.01 = 1.00$ | 🔴 *Descartado* |

*El "núcleo" filtrado queda formado únicamente por `{café, té, agua}`.*

---

### **Paso 2: Re-normalización y Muestreo**

Las probabilidades del grupo seleccionado ($0.58 + 0.26 + 0.11 = 0.95$) se dividen entre $0.95$ para que sumen el $100\%$:

* **café:** $61.1\%$
* **té:** $27.4\%$
* **agua:** $11.5\%$

Se realiza un muestreo multinomial sobre este trío, garantizando variabilidad sin caer en palabras absurdas.

---

### **¿Por qué Top-p es conceptualmente superior a Top-K?**

Top-p se **adapta dinámicamente** a la incertidumbre del modelo en cada paso de generación:

1. **Cuando el modelo está muy seguro (pico de probabilidad alto):**
* Supongamos que *"café"* tuviera un **$92\%$** de probabilidad por sí solo.
* **Top-p ($p=0.90$):** Se detiene en la primera palabra ($92\% \ge 90\%$) y solo muestrea *"café"*, comportándose de forma determinista como **Greedy Search**.
* **Top-K ($K=3$):** Obligaría rígidamente a meter $2$ palabras más, arriesgándose a elegir una opción mediocre.


2. **Cuando el modelo está dudoso (distribución plana):**
* Si las probabilidades estuvieran repartidas entre 10 palabras diferentes (10% cada una).
* **Top-p ($p=0.90$):** Ampliará automáticamente la lista a las **9 mejores palabras** para cubrir el $90\%$.
* **Top-K ($K=3$):** Cortaría prematuramente en $3$ palabras, dejando por fuera opciones muy válidas.



---

### **Resumen Comparativo de las 3 Estrategias**

| Estrategia | ¿Cómo selecciona los candidatos? | Tamaño de la lista | Ventaja principal |
| --- | --- | --- | --- |
| **Greedy Search** | Elige únicamente la opción #1. | Siempre 1 token. | Rápido y determinista. Ideal para datos estructurados o código. |
| **Top-K Sampling** | Toma los $K$ primeros tokens ordenados. | Fijo ($K$). | Evita *tokens* raros pero es inflexible. |
| **Top-p (Nucleus)** | Suma probabilidades hasta alcanzar el umbral $p$. | **Dinámico** (cambia en cada palabra). | Máxima naturalidad y fluidez en texto creativo/chat. |

In [ ]:
# P=98 cuantos tokens se necesitan para llegar al 98%? si el tokens tiene una P de 99 solo tendra una probabilidad de ser 1 token

### Conclusión

No hay un método formal único para elegir un token. Si quieres tener una conversación fluida con un modelo, es conveniente añadir un poco más de aleatoriedad en la selección de fichas para que no suene rígido ni repetitivo. Por otro lado, si usas un LLM para redactar un documento técnico o jurídico, no querrás que la generación se vuelva incontrolable y empiece a incluir fragmentos aleatorios con apenas una ligera relación con tu objetivo real.

En cambio, si buscas que el modelo escriba cuentos infantiles y tenga libertad narrativa, los giros inesperados en la trama resultan sumamente ingeniosos y divertidos. Pero si el modelo debe proporcionar información histórica precisa, lo último que deseas es que empiece a hacer afirmaciones erróneas o inventadas simplemente porque el token correcto fue descartado durante el muestreo probabilístico.